# Agent Observer (Phase 4)

This notebook implements the **Observer agent**.

Phase 4 scope:
- Subscribe to Trigger person-state topic
- Compute pairwise distance between infected and susceptible persons
- Publish exposure candidates to observer exposure topic
- Publish city snapshot metrics for each completed step

Observer only detects candidates; it does not change health states.

In [1]:
# Cell purpose: Import modules, load config, and define Phase 4 topic contracts.
from __future__ import annotations

import json
import math
from collections import defaultdict

import simulated_city.mqtt as mqtt
from simulated_city.config import load_config

cfg = load_config()
sim_cfg = cfg.simulation
mqtt_cfg = cfg.mqtt

if sim_cfg is None:
    raise ValueError("Phase 4 requires a 'simulation' section in config.yaml")

base_topic = mqtt_cfg.base_topic
topic_trigger_state = f"{base_topic}/pandemic/trigger/person_state"
topic_observer_exposure = f"{base_topic}/pandemic/observer/exposure_event"
topic_observer_snapshot = f"{base_topic}/pandemic/observer/city_snapshot"

infection_radius_m = sim_cfg.infection_radius_m
population_size = sim_cfg.population_size
min_people_for_step = max(1, int(population_size * 0.9))

print(f"Observer config loaded. population_size={population_size}, infection_radius_m={infection_radius_m}")
print(f"Step completion threshold: {min_people_for_step}/{population_size} persons")
print(f"Subscribe topic: {topic_trigger_state}")
print(f"Publish topics: {topic_observer_exposure}, {topic_observer_snapshot}")

Observer config loaded. population_size=300, infection_radius_m=2.0
Step completion threshold: 270/300 persons
Subscribe topic: simulated-city/pandemic/trigger/person_state
Publish topics: simulated-city/pandemic/observer/exposure_event, simulated-city/pandemic/observer/city_snapshot


In [2]:
# Cell purpose: Define distance + step-processing logic and in-memory message buffers.
def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    radius_m = 6_371_000.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return radius_m * c


step_people: dict[int, dict[str, dict]] = defaultdict(dict)
processed_steps: set[int] = set()
steps_processed = 0
exposure_events_published = 0
city_snapshots_published = 0
client = None


def process_step(step: int, people_by_id: dict[str, dict]) -> tuple[int, int, int, int]:
    global city_snapshots_published

    persons = list(people_by_id.values())
    infected = [p for p in persons if p.get("health_status") == "infected"]
    susceptible = [p for p in persons if p.get("health_status") == "susceptible"]
    recovered = [p for p in persons if p.get("health_status") == "recovered"]

    exposure_count = 0
    for source in infected:
        for target in susceptible:
            distance_m = haversine_m(source["lat"], source["lon"], target["lat"], target["lon"])
            within_radius = distance_m <= infection_radius_m
            if not within_radius:
                continue

            exposure_payload = {
                "step": step,
                "ts": source["ts"],
                "source_id": source["person_id"],
                "target_id": target["person_id"],
                "distance_m": round(distance_m, 3),
                "within_radius": True,
            }
            mqtt.publish_json_checked(client, topic_observer_exposure, exposure_payload)
            exposure_count += 1

    snapshot_payload = {
        "step": step,
        "ts": persons[0]["ts"] if persons else "",
        "population": len(persons),
        "infected_count": len(infected),
        "susceptible_count": len(susceptible),
        "recovered_count": len(recovered),
        "active_exposures": exposure_count,
    }
    mqtt.publish_json_checked(client, topic_observer_snapshot, snapshot_payload)
    city_snapshots_published += 1

    return exposure_count, len(infected), len(susceptible), len(recovered)

In [3]:
# Cell purpose: Connect to MQTT, subscribe to Trigger state topic, and process completed simulation steps.
client = mqtt.connect_mqtt(mqtt_cfg, client_id_suffix="observer")
print(f"Connected to MQTT broker at {mqtt_cfg.host}:{mqtt_cfg.port}")


def on_message(client_obj, userdata, msg):
    global steps_processed, exposure_events_published

    try:
        payload = json.loads(msg.payload.decode())
        step = int(payload["step"])

        person_id = str(payload["person_id"])

        step_people[step][person_id] = payload

        candidate_steps = sorted(s for s in step_people.keys() if s not in processed_steps)
        for candidate_step in candidate_steps:
            people_count = len(step_people[candidate_step])
            has_min_people = people_count >= min_people_for_step
            has_newer_step = any(s > candidate_step for s in step_people.keys())
            has_full_people = people_count >= population_size

            if not has_min_people:
                continue
            if not has_full_people and not has_newer_step:
                continue

            exposures, infected_count, susceptible_count, recovered_count = process_step(candidate_step, step_people[candidate_step])
            steps_processed += 1
            exposure_events_published += exposures
            processed_steps.add(candidate_step)

            if candidate_step < 3 or candidate_step % 100 == 0:
                print(
                    f"processed step={candidate_step:04d} persons={people_count} "
                    f"exposures={exposures} infected={infected_count} "
                    f"susceptible={susceptible_count} recovered={recovered_count}"
                )

            del step_people[candidate_step]

    except Exception as exc:
        print(f"Observer message handling error: {exc}")


client.on_message = on_message
client.subscribe(topic_trigger_state)
print(f"Subscribed to {topic_trigger_state}")
print("Observer is now listening. Run Trigger notebook publish cell to feed this agent.")

Connected to MQTT broker at 127.0.0.1:1883
Subscribed to simulated-city/pandemic/trigger/person_state
Observer is now listening. Run Trigger notebook publish cell to feed this agent.


processed step=0000 persons=300 exposures=0 infected=5 susceptible=295 recovered=0
processed step=0001 persons=300 exposures=0 infected=5 susceptible=295 recovered=0
processed step=0002 persons=300 exposures=0 infected=5 susceptible=295 recovered=0
processed step=0500 persons=300 exposures=0 infected=5 susceptible=295 recovered=0
processed step=0700 persons=300 exposures=0 infected=5 susceptible=295 recovered=0


In [4]:
# Cell purpose: Show observer summary counters and disconnect cleanly when finished.
print(
    f"Observer summary: steps_processed={steps_processed}, "
    f"exposure_events_published={exposure_events_published}, "
    f"city_snapshots_published={city_snapshots_published}, "
    f"pending_steps={len(step_people)}"
)

connector = getattr(client, "_simulated_city_connector", None)
if connector is not None:
    connector.disconnect()
    print("Disconnected from MQTT broker.")
else:
    print("No connector reference found; client disconnect skipped.")

Observer summary: steps_processed=106, exposure_events_published=0, city_snapshots_published=106, pending_steps=94
Disconnected from MQTT broker.
